# Riesgo y predicción cuantitativa: factores asociados al mal riesgo crediticio

**Curso:** MCC002 - Probabilidad y Estadística Computacional 

**Grupo 5:** 
- Armando Castro Chaupis 
- Henry Sánchez Alvarado 
- Alex Segura Núñez

**Pregunta principal:** ¿Qué factores explican la probabilidad de que un solicitante sea clasificado como mal riesgo crediticio?

**Preguntas secundarias:**

1. Proporción global de malos créditos y su IC 95 %.
2. ¿La tasa de mal crédito difiere según el propósito del préstamo?
3. ¿La duración y el monto del crédito difieren entre buenos y malos créditos?
4. ¿Qué variables están asociadas con mayor riesgo crediticio?
5. ¿Cómo cambia la clasificación al modificar el umbral de decisión?

## 1. Importación de librerias

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import statsmodels.api as sm
import statsmodels.formula.api as smf
import sklearn
from scipy import stats 

### Establecemos valor de semilla
SEED = 2026
np.random.seed(SEED)

### Estilo de gráficos para matplotlib
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 12


## 2. Carga de dataset

import os, io, zipfile, urllib.request

UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/144/statlog+german+credit+data.zip"
CANDIDATOS = ["data/german.data", "german.data"]

ruta_datos = next((p for p in CANDIDATOS if os.path.exists(p)), None)

if ruta_datos is None:
    # Descarga documentada desde la fuente oficial (solo si no existe copia local)
    os.makedirs("data", exist_ok=True)
    print("Descargando dataset desde UCI...")
    with urllib.request.urlopen(UCI_ZIP_URL) as r:
        zf = zipfile.ZipFile(io.BytesIO(r.read()))
        zf.extract("german.data", path="data")
    ruta_datos = "data/german.data"

# Nombres de columnas según german.doc (atributos 1..20 + clase)
columnas = [
    "estado_cuenta", "duracion", "historial_credito", "proposito", "monto",
    "ahorros", "empleo_actual", "tasa_cuota", "estatus_personal_sexo",
    "otros_deudores", "anios_residencia", "propiedad", "edad",
    "otros_planes_pago", "vivienda", "n_creditos_banco", "trabajo",
    "n_dependientes", "telefono", "trabajador_extranjero", "clase",
]

df = pd.read_csv(ruta_datos, sep=" ", header=None, names=columnas)
print(f"Archivo cargado: {ruta_datos}")
print(f"Dimensiones: {df.shape[0]} observaciones x {df.shape[1]} columnas")
df.head()

### 3.1 Diccionario de variables

Construido a partir del dataset. Las 13 variables cualitativas usan códigos `A**`. 
Abajo se documenta el significado de cada código y se crea un mapeo de etiquetas.

| # | Variable (nombre en el notebook) | Tipo | Descripción |
|---|---|---|---|
| 1 | `estado_cuenta` | Categórica ordinal | Estado de la cuenta corriente: A11 (< 0 DM), A12 (0–200 DM), A13 (≥ 200 DM), A14 (sin cuenta) |
| 2 | `duracion` | Numérica (meses) | Duración del crédito |
| 3 | `historial_credito` | Categórica | A30–A34: de "sin créditos/todo pagado" a "cuenta crítica/créditos en otros bancos" |
| 4 | `proposito` | Categórica nominal | A40–A410: auto nuevo/usado, mobiliario, radio/TV, electrodomésticos, reparaciones, educación, reentrenamiento, negocio, otros |
| 5 | `monto` | Numérica (DM) | Monto del crédito |
| 6 | `ahorros` | Categórica ordinal | A61–A65: nivel de ahorros/bonos (A65 = desconocido/sin cuenta) |
| 7 | `empleo_actual` | Categórica ordinal | A71–A75: antigüedad en el empleo actual |
| 8 | `tasa_cuota` | Numérica (1–4) | Cuota como % del ingreso disponible |
| 9 | `estatus_personal_sexo` | Categórica | A91–A94: estado civil y sexo (composición histórica del dataset) |
| 10 | `otros_deudores` | Categórica | A101 ninguno, A102 co-solicitante, A103 garante |
| 11 | `anios_residencia` | Numérica (1–4) | Años en la residencia actual |
| 12 | `propiedad` | Categórica | A121 inmueble … A124 sin propiedad conocida |
| 13 | `edad` | Numérica (años) | Edad del solicitante |
| 14 | `otros_planes_pago` | Categórica | A141 banco, A142 tiendas, A143 ninguno |
| 15 | `vivienda` | Categórica | A151 alquilada, A152 propia, A153 gratuita |
| 16 | `n_creditos_banco` | Numérica | N.º de créditos existentes en este banco |
| 17 | `trabajo` | Categórica ordinal | A171–A174: de no calificado/no residente a directivo/independiente |
| 18 | `n_dependientes` | Numérica | Personas a cargo |
| 19 | `telefono` | Binaria | A191 no, A192 sí (registrado) |
| 20 | `trabajador_extranjero` | Binaria | A201 sí, A202 no |
| 21 | `clase` | **Variable respuesta** | 1 = buen riesgo, 2 = mal riesgo |